# Part 2: CNN-Based Manufacturing Defect Classification
## Computer Vision Problem Formulation and CNN Prototype

**Dataset:** Synthetic Manufacturing Defect Image Dataset  
**Classes:** `normal`, `scratch`, `dent`, `stain`  
**Goal:** Train a CNN to classify product surface images into one of the four defect categories.

---

## Task 1 — Problem Identification

**Selected Problem Type: Image Classification**

Each image in the dataset carries a single label describing the type of defect (or its absence) across the entire surface. The model must predict which of the four mutually exclusive categories a given image belongs to.

- **Not object detection** – we do not need to locate *where* the defect is; we only classify the image.
- **Not segmentation** – we do not assign class labels to individual pixels.
- **Image classification** is the correct framing: input = image → output = one class label.

## Task 2 — Dataset Exploration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ── Configuration ─────────────────────────────────────────────────────────────
IMG_DIR   = "images"           # root folder: images/normal/, images/scratch/, ...
LABELS_CSV = "labels.csv"
CLASSES   = ["normal", "scratch", "dent", "stain"]
IMG_SIZE  = 64                 # resize target (px)
SEED      = 42
np.random.seed(SEED)

In [ ]:
# ── Load labels ───────────────────────────────────────────────────────────────
df = pd.read_csv(LABELS_CSV)
print("Dataset shape:", df.shape)
print()
print(df.head(8))

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
counts = df["class"].value_counts().reindex(CLASSES)
print("Images per class:")
print(counts.to_string())
print()
print(f"Total images : {len(df)}")
print(f"Num classes  : {df['class'].nunique()}")
print(f"Balanced     : {counts.std() < 1.0}")

In [ ]:
# ── Visualise sample images ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Sample Images Per Class", fontsize=16, fontweight="bold")

for col, cls in enumerate(CLASSES):
    cls_files = df[df["class"] == cls]["filename"].values
    for row in range(2):
        path = cls_files[row]
        img  = Image.open(path).resize((IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(img)
        if row == 0:
            axes[row, col].set_title(cls, fontsize=12, fontweight="bold")
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig("results/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Image dimensions ──────────────────────────────────────────────────────────
sample_path = df["filename"].iloc[0]
sample_img  = Image.open(sample_path)
print(f"Original image size : {sample_img.size}")
print(f"Mode (colour space) : {sample_img.mode}")
print(f"Resized to          : ({IMG_SIZE}, {IMG_SIZE})")

## Task 3 — Image Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

# ── Load all images into NumPy arrays ─────────────────────────────────────────
X, y = [], []
label_map = {cls: idx for idx, cls in enumerate(CLASSES)}

for _, row in df.iterrows():
    img = Image.open(row["filename"]).resize((IMG_SIZE, IMG_SIZE)).convert("RGB")
    X.append(np.array(img))
    y.append(label_map[row["class"]])

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("Raw X shape :", X.shape)
print("y shape     :", y.shape)
print("Pixel range  [min, max]:", X.min(), X.max())

In [ ]:
# ── 1. Normalize pixel values to [0, 1] ───────────────────────────────────────
X = X / 255.0
print("After normalization [min, max]:", X.min(), X.max())

# ── 2. Train / Test split (80/20, stratified) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# ── 3. Train / Validation split (85/15 of training, stratified) ───────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

print(f"\nSplit summary:")
print(f"  Train       : {len(X_train)} images")
print(f"  Validation  : {len(X_val)} images")
print(f"  Test        : {len(X_test)} images")
print(f"  Total       : {len(X_train)+len(X_val)+len(X_test)} images")

In [ ]:
# ── 4. Data augmentation (defined as Keras layers, applied during training) ────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
], name="augmentation")

# Preview augmented images
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle("Augmentation preview (same source image)", fontsize=13, fontweight="bold")
sample_image = X_train[0:1]  # shape (1, 64, 64, 3)
for i in range(5):
    augmented = data_augmentation(sample_image, training=True)
    axes[i].imshow(augmented[0])
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## Task 4 — CNN Model Creation

In [ ]:
tf.random.set_seed(SEED)

model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # ── Augmentation (active during training only) ──────────────────────────
    data_augmentation,

    # ── Convolutional Block 1 ───────────────────────────────────────────────
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    # ── Convolutional Block 2 ───────────────────────────────────────────────
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    # ── Convolutional Block 3 ───────────────────────────────────────────────
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    # ── Classifier head ─────────────────────────────────────────────────────
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.50),
    layers.Dense(len(CLASSES), activation="softmax"),   # Output layer
], name="ManufacturingDefectCNN")

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model compiled.")

## Task 5 — Model Training & Evaluation

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ── Accuracy and Loss Curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training History", fontsize=15, fontweight="bold")

ax = axes[0]
ax.plot(history.history["accuracy"],     label="Train", linewidth=2.5, color="#1565C0")
ax.plot(history.history["val_accuracy"], label="Val",   linewidth=2.5, color="#E65100", linestyle="--")
ax.set_title("Accuracy", fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

ax = axes[1]
ax.plot(history.history["loss"],     label="Train", linewidth=2.5, color="#1565C0")
ax.plot(history.history["val_loss"], label="Val",   linewidth=2.5, color="#E65100", linestyle="--")
ax.set_title("Loss", fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/accuracy_loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Test Evaluation ───────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test Loss     : {test_loss:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=CLASSES, yticklabels=CLASSES, linewidths=0.5,
            annot_kws={"size": 14, "weight": "bold"})
ax.set_title(f"Confusion Matrix  (Test Acc: {test_acc:.1%})", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Label", fontweight="bold")
ax.set_ylabel("True Label", fontweight="bold")
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
# ── Sample Predictions ────────────────────────────────────────────────────────
n_show = 12
idxs   = np.random.choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Sample Predictions on Test Set\n(Green = Correct | Red = Incorrect)",
             fontsize=14, fontweight="bold")

for i, idx in enumerate(idxs):
    ax = axes[i // 4, i % 4]
    ax.imshow(X_test[idx])
    true_cls = CLASSES[y_test[idx]]
    pred_cls = CLASSES[y_pred[idx]]
    ok = true_cls == pred_cls
    color = "#2e7d32" if ok else "#c62828"
    ax.set_title(f"{'✓' if ok else '✗'}  True: {true_cls}\n   Pred: {pred_cls}",
                 fontsize=9, color=color, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("sample_predictions/prediction_outputs.png", dpi=120, bbox_inches="tight")
plt.show()

## Task 6 — CNN Concept Explanation

### What is Convolution?
A small learnable filter (e.g. 3×3 matrix of weights) slides across the entire image one position at a time, computing a dot product at each location. The result is a **feature map** — a 2-D representation highlighting where a specific pattern (edge, curve, texture) is present. By stacking many filters (32, 64, 128), the network learns a rich vocabulary of visual patterns at every layer.

**Key insight:** the same filter is shared across all positions → far fewer parameters than a fully connected layer, and the model learns patterns regardless of *where* they appear in the image.

---

### Why is Pooling Used?
`MaxPooling(2×2)` takes the maximum value in each 2×2 block, halving both width and height. This:
1. **Reduces spatial size** → fewer computations in deeper layers.
2. **Builds positional invariance** → a scratch shifted by 1-2 pixels still produces the same pooled representation.
3. **Forces abstraction** → deeper layers learn "there is a scratch somewhere in this region" rather than "there is a scratch at pixel (32,15)".

---

### Why is ReLU Commonly Used?
`ReLU(x) = max(0, x)` is the default activation because:
- **No vanishing gradients** (for positive values the gradient = 1), enabling training of deep networks.
- **Sparsity** — many neurons output 0, making activations sparse and representations efficient.
- **Speed** — a simple comparison, orders of magnitude faster than `sigmoid` or `tanh` (which require exponentials).

---

### Why CNNs Beat Feed-Forward Networks for Images?
| Issue | Fully Connected Net | CNN |
|---|---|---|
| Parameter count | Quadratic in image size | Roughly constant (shared filters) |
| Translation sensitivity | Pixel shift → completely different vector | Same filter detects pattern anywhere |
| Spatial structure | Flattening destroys 2-D layout | Convolution preserves neighbourhood relationships |
| Sample efficiency | Needs millions of images | Learns from hundreds due to parameter sharing |

---

## Task 7 — Business Use Case: Manufacturing Quality Inspection

### Scenario
An electronics manufacturer produces 10,000 PCB (printed circuit board) housings per day. Defective units (scratches, dents, stains) must be detected and rejected before packaging. Manual inspection at this scale is infeasible.

### CNN-Based Solution
An inline camera captures each unit as it moves along the conveyor. A CNN model (like the one built here) classifies the surface in real time (< 30 ms/image on a GPU edge device). Defective items trigger an ejector arm.

### Workflow
```
Camera → Pre-process → CNN Inference → Decision
  ↓           ↓              ↓              ↓
Image     Resize+Norm   normal / ...   Pass / Reject
```

### Benefits
| KPI | Before (Manual) | After (CNN) |
|---|---|---|
| Throughput | 500 units/hr | 5,000+ units/hr |
| Consistency | Varies (fatigue) | 100% consistent |
| False-negative rate | ~15% | < 5% |
| Traceability | Paper logs | Timestamped image archive |
| Cost | High labour | Hardware CapEx, minimal OpEx |

### Scalability
The same model can be retrained for any product category (automotive parts, food packaging, pharmaceutical blister packs) with minimal effort — just relabel a new image dataset and fine-tune.